# Notebook 3 — Carga de Datos a PostgreSQL
## Caso: Predicción de Default en Solicitudes de Préstamos

### Objetivo
Cargar los datos validados a PostgreSQL en dos tablas:
- **loans_clean** → registros válidos
- **loans_error** → registros rechazados

### Etapas
1. Cargar datasets validados
2. Conectarse a PostgreSQL
3. Cargar registros válidos a tabla loans_clean
4. Cargar registros inválidos a tabla loans_error
5. Verificar carga

In [9]:
import pandas as pd
import os
import logging
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [10]:
# Crear carpeta logs si no existe
os.makedirs("/workspaces/dataops-loan-pipeline/logs", exist_ok=True)

# Resetear el logger para que tome la nueva configuración
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configurar el sistema de logs para este notebook
logging.basicConfig(
    filename="/workspaces/dataops-loan-pipeline/logs/load_database.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Registrar inicio del proceso
logging.info("Inicio del proceso de carga a PostgreSQL - Pipeline DataOps")
print("✅ Logger configurado correctamente")

✅ Logger configurado correctamente


In [3]:
# Cargar las variables de entorno desde el archivo .env
load_dotenv("../.env")

# Leer las credenciales de la base de datos desde el .env
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Crear la cadena de conexión a PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Crear el engine de conexión
engine = create_engine(connection_string)

# Verificar que la conexión funciona
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
        logging.info("Conexión a PostgreSQL exitosa - Base de datos: loans_db")
        print("✅ Conexión a PostgreSQL exitosa")
except Exception as e:
    logging.error(f"Error al conectar a loans_db: {e}")
    print(f"❌ Error al conectar: {e}")

✅ Conexión a PostgreSQL exitosa


In [4]:
# Leer los datasets generados en el Notebook 2
df_validos = pd.read_csv("../data/processed/loans_validos.csv")
df_invalidos = pd.read_csv("../data/processed/loans_invalidos.csv")

# Registrar en el log
logging.info(f"loans_validos.csv cargado desde data/processed: {len(df_validos)} registros aptos para carga")
logging.info(f"loans_invalidos.csv cargado desde data/processed: {len(df_invalidos)} registros rechazados para auditoría")
print(f"✅ Registros válidos: {len(df_validos)}")
print(f"✅ Registros inválidos: {len(df_invalidos)}")

✅ Registros válidos: 44993
✅ Registros inválidos: 7


In [5]:
# Cargar registros válidos a tabla loans_clean
try:
    df_validos.to_sql("loans_clean", engine, if_exists="replace", index=False)
    logging.info(f"Tabla loans_clean cargada exitosamente en PostgreSQL: {len(df_validos)} registros válidos para análisis")
    print(f"✅ Tabla loans_clean cargada: {len(df_validos)} registros")
except Exception as e:
    logging.error(f"Error al cargar tabla loans_clean en PostgreSQL: {e}")
    print(f"❌ Error al cargar loans_clean: {e}")

# Cargar registros inválidos a tabla loans_error
try:
    df_invalidos.to_sql("loans_error", engine, if_exists="replace", index=False)
    logging.info(f"Tabla loans_error cargada exitosamente en PostgreSQL: {len(df_invalidos)} registros rechazados para auditoría")
    print(f"✅ Tabla loans_error cargada: {len(df_invalidos)} registros")
except Exception as e:
    logging.error(f"Error al cargar tabla loans_error en PostgreSQL: {e}")
    print(f"❌ Error al cargar loans_error: {e}")

✅ Tabla loans_clean cargada: 44993 registros
✅ Tabla loans_error cargada: 7 registros


In [6]:
# Verificar tabla loans_clean
df_check_clean = pd.read_sql("SELECT COUNT(*) as total FROM loans_clean", engine)
logging.info(f"Verificación exitosa - Tabla loans_clean en PostgreSQL: {df_check_clean['total'][0]} registros")
print(f"✅ loans_clean en PostgreSQL: {df_check_clean['total'][0]} registros")

# Verificar tabla loans_error
df_check_error = pd.read_sql("SELECT COUNT(*) as total FROM loans_error", engine)
logging.info(f"Verificación exitosa - Tabla loans_error en PostgreSQL: {df_check_error['total'][0]} registros")
print(f"✅ loans_error en PostgreSQL: {df_check_error['total'][0]} registros")

# Mostrar primeras filas de loans_clean
print()
print("📊 Primeras filas de loans_clean:")
pd.read_sql("SELECT * FROM loans_clean LIMIT 5", engine)

✅ loans_clean en PostgreSQL: 44993 registros
✅ loans_error en PostgreSQL: 7 registros

📊 Primeras filas de loans_clean:


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22,female,Master,71948,0,RENT,35000,PERSONAL,16.02,0.49,3,561,No,1
1,21,female,High School,12282,0,OWN,1000,EDUCATION,11.14,0.08,2,504,Yes,0
2,25,female,High School,12438,3,MORTGAGE,5500,MEDICAL,12.87,0.44,3,635,No,1
3,23,female,Bachelor,79753,0,RENT,35000,MEDICAL,15.23,0.44,2,675,No,1
4,24,male,Master,66135,1,RENT,35000,MEDICAL,14.27,0.53,4,586,No,1


In [7]:
logging.info("Fin del proceso de carga a PostgreSQL - Pipeline DataOps completado exitosamente")
logging.info(f"Resumen final - loans_clean: {len(df_validos)} registros | loans_error: {len(df_invalidos)} registros | Total: {len(df_validos) + len(df_invalidos)}")
print("🎉 Pipeline DataOps completado exitosamente")
print()
print("Resumen final:")
print(f"  ✅ Registros cargados en loans_clean: {len(df_validos)}")
print(f"  ❌ Registros rechazados en loans_error: {len(df_invalidos)}")
print(f"  📊 Total procesado: {len(df_validos) + len(df_invalidos)}")

🎉 Pipeline DataOps completado exitosamente

Resumen final:
  ✅ Registros cargados en loans_clean: 44993
  ❌ Registros rechazados en loans_error: 7
  📊 Total procesado: 45000


In [8]:
import time

# KPI 1: Tasa de registros inválidos
tasa_invalidos = (len(df_invalidos) / (len(df_validos) + len(df_invalidos))) * 100

# KPI 2: Tasa de completitud
tasa_completitud = (len(df_validos) / (len(df_validos) + len(df_invalidos))) * 100

# KPI 3: Total registros procesados
total_procesado = len(df_validos) + len(df_invalidos)

# Registrar KPIs en el log
logging.info(f"KPI - Total registros procesados: {total_procesado}")
logging.info(f"KPI - Tasa de completitud: {tasa_completitud:.2f}%")
logging.info(f"KPI - Tasa de registros inválidos: {tasa_invalidos:.2f}%")

# Mostrar KPIs
print("📊 KPIs de Monitoreo del Pipeline")
print("=" * 40)
print(f"Total registros procesados:  {total_procesado}")
print(f"Tasa de completitud:         {tasa_completitud:.2f}%")
print(f"Tasa de registros inválidos: {tasa_invalidos:.2f}%")
print(f"Registros válidos cargados:  {len(df_validos)}")
print(f"Registros rechazados:        {len(df_invalidos)}")

# Alerta si la tasa de inválidos supera el 5%
if tasa_invalidos > 5:
    logging.warning(f"⚠️ ALERTA - Tasa de inválidos alta: {tasa_invalidos:.2f}% supera el umbral del 5%")
    print()
    print("⚠️ ALERTA: Tasa de registros inválidos supera el 5%")
else:
    logging.info(f"KPI - Tasa de inválidos aceptable: {tasa_invalidos:.2f}% dentro del umbral del 5%")
    print()
    print("✅ Tasa de inválidos dentro del rango aceptable")

📊 KPIs de Monitoreo del Pipeline
Total registros procesados:  45000
Tasa de completitud:         99.98%
Tasa de registros inválidos: 0.02%
Registros válidos cargados:  44993
Registros rechazados:        7

✅ Tasa de inválidos dentro del rango aceptable
